# 05 — Probe de agua y prompts automáticos

**Solo etapa 5. Ejecución por el usuario exclusivamente en Google Colab con GPU.** En la PC solo desarrollo y pruebas sintéticas CPU sin modelos ni entrenamiento. Reutiliza las etapas 1–4: revisiones autorizadas, RGB/normalización por régimen, geometría, métricas y persistencia.

Train/val: official 252/89, leave_paraguay_out 213/75. Probes, scaler, features y decisiones separados. No test, Bolivia ni Paraguay reservado. No decoder, campaña final, A1–A3 ni demo. La etapa queda abierta hasta revisar notebook con salidas, JSON y artefactos reales.

Lee `docs/project/validacion_etapa5.md` del paquete antes de ejecutar. Se mantiene la salvedad de licencia del dataset.


## Protocolo previo y presupuesto

DINOv3 SAT congelado, final LayerNorm, sin CLS/registros, grilla 32×32×1024. Targets por fracción de agua sobre píxeles válidos de etiqueta: ≥0.50 agua, ≤0.05 no agua, resto/sin válidos ignorado. Scaler y regresión logística se ajustan solo con train. Cuatro probes: C=[0.1,1], class_weight=[none,balanced], todos los parches inequívocos, solver CUDA LBFGS. Ocho filas de puntos al cruzar umbral positivo [0.5,0.7]. Selección por IoU agua acumulada de segmentación val; empate conserva primer candidato. Convergencia y diagnósticos auxiliares quedan registrados.

Mapa bilineal nativo con bordes replicados; conectividad 8, mínimo 16 píxeles; hasta 3 positivos y 3 negativos por componente, distancia mínima 16 píxeles originales. Negativos globales con p≤0.1. Empates deterministas, sin relajación; cero componentes produce vacío sin SAM. SAM congelado: mayor IoU predicha y unión. Inferencia recibe solo imagen/features/validez de entrada y configuración; los lectores de etiquetas se deshabilitan durante las llamadas automáticas. Se leen después para métricas.

Malla SAM oficial 32×32 (batch 64, crop completo, sin m2m), IoU predicha >0.8, estabilidad ≥0.95, offset 1, NMS 0.7; aceptación por media DINO sobre máscara∩validez de entrada [0.5,0.7], calibrada en val con probe ya fijado. No se llama agua a toda máscara agnóstica.

**NDWI aplazado antes de las comparaciones.** El probe actual depende de etiquetas train y calibración manual val. No se afirma que el sistema sea libre de etiquetas.

Almacenar al menos varios GiB adicionales en Drive; solo las features sin comprimir son ~2.46 GiB. DINO se libera antes del ajuste y SAM se carga después. Ante OOM se conserva el fallo, sin modificar búsqueda/resoluciones automáticamente.


## 1. Paquete y entorno

En la PC ejecuta `scripts/export_stage5_bundle.py` con el Python del venv de validación. Sube el ZIP a MyDrive y copia su SHA-256. Usa la raíz histórica de Drive con raw/stage2/regimes y caché autorizada de modelos. Selecciona GPU en Colab. `HF_TOKEN` opcional en Secrets; nunca lo escribas en celdas o salidas.


In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'
REPO = Path('/content/auto-prompt-sam2-stage5')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage5.zip')
EXPECTED_BUNDLE_SHA256 = '5f50481dee90176390604c037e3d5768e666e1dcce50106a79713c840ff47578'  # Copiar salida de export_stage5_bundle.py.
assert len(EXPECTED_BUNDLE_SHA256) == 64, 'Configura el SHA-256 del paquete.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        assert (REPO / item.filename).resolve().is_relative_to(REPO.resolve())
    archive.extractall(REPO)
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
# Token opcional para descarga autorizada, nunca imprimirlo ni incluirlo en informes.
from google.colab import userdata
try:
    child_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print('HF_TOKEN no configurado; se usara la cache autorizada si existe.')
LOG_DIR = Path(PERSIST_ROOT) / 'stage5/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
def command(args, log_name):
    with (LOG_DIR / log_name).open('w', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; conservar salida y log {log_name}')
if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py')], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 182.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 41.5 MB/s  0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 40.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 135.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 183.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 60.6 MB/s  0:00:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 117.5 M

## 2. Extraer, ajustar y calibrar en Colab

Ejecuta `prepare` solo una vez. Genera shards train/val, cuatro probes independientes por régimen, ocho filas de puntos y dos de malla. Cada unidad se guarda antes de continuar. **Si hay interrupción**, detén primero la sesión/proceso anterior y cambia MODE a `resume` conservando exactamente paquete, SHA, raíz y configuración. Reinicia únicamente unidades incompletas. No uses `prepare` como sustituto de `recover`.

Las recomendaciones y NDWI ya quedaron fijados en el paquete antes de ver resultados de cualquiera de los regímenes. Conserva fallos y candidatos sin convergencia; no amplíes la búsqueda al observar el rendimiento.


In [ ]:
MODE = 'prepare'  # 'resume' solo tras detener al escritor anterior.
assert MODE in ('prepare', 'resume')
command([str(PYTHON), str(REPO / 'scripts/run_stage5.py'), MODE, '--root', PERSIST_ROOT], MODE + '.log')


fatal: not a git repository (or any of the parent directories): .git
No broken requirements found.
 
 test_canonical_hash (test_stage1.PersistenceTests.test_canonical_hash) ... ok
test_colab_temporary_root_blocked (test_stage1.PersistenceTests.test_colab_temporary_root_blocked) ... ok
test_each_identity_change_is_rejected (test_stage1.PersistenceTests.test_each_identity_change_is_rejected) ... ok
test_export_reports_absent_access_instead_of_claiming_authorization (test_stage1.PersistenceTests.test_export_reports_absent_access_instead_of_claiming_authorization) ... ok
test_export_uses_saved_identity_and_only_verified_json (test_stage1.PersistenceTests.test_export_uses_saved_identity_and_only_verified_json) ... ok
test_head_never_logs_token_or_follows_redirect (test_stage1.PersistenceTests.test_head_never_logs_token_or_follows_redirect) ... ok
test_incomplete_and_corrupt_generation_preserve_previous (test_stage1.PersistenceTests.test_incomplete_and_corrupt_generation_preserve_previous) .

## 3. Inspeccionar salidas de validación

Estas son calibraciones de desarrollo, no resultados finales. Inspecciona balances/ignorados, convergencia, ocho candidatos completos, presupuesto real de puntos, máscaras vacías y las selecciones de cada régimen. Un mal resultado se conserva. El marcador complete acredita cómputo; no cierra la etapa.


In [ ]:
EVIDENCE = Path(PERSIST_ROOT) / 'stage5/evidence'
RUNS = sorted(p for p in EVIDENCE.iterdir() if p.is_dir() and (p / 'validation.json').is_file())
assert RUNS, 'No hay prepare completo: conservar logs y failure.json.'
RUN_DIR = RUNS[-1]
REPORT = json.loads((RUN_DIR / 'validation.json').read_text())
print('run_id:', REPORT['run_id'], 'status:', REPORT['status'])
for regime, value in REPORT['regimes'].items():
    print(regime, 'patch_balance:', value['patch_balance'])
    for row in value['selection']['search_rows']:
        print(row['candidate_id'], row['candidate'], row['positive_threshold'], 'eligible:', row['eligible'],
              'val IoU:', row['metrics']['pooled']['values']['water_iou'], 'fit:', row['fit'])
    print('POINTS SELECTED:', value['selection']['selected']['candidate_id'])
    print('GRID SELECTED:', value['grid_selection']['selected'])
    print('RECONSTRUCTION:', value['disk_reconstruction'])
print('NDWI:', REPORT['ndwi'])
print('ZIP compacto:', RUN_DIR.with_suffix('.zip'))
print('Features completas permanecen en Drive; ZIP solo incluye primer shard val de cada régimen.')


run_id: 20261029T194653Z_8073c068fa_ed6583db status: computation_passed_pending_external_review
leave_paraguay_out patch_balance: {'train': {'ignored': 42681, 'no_valid': 24377, 'nonwater': 158771, 'water': 16660}, 'val': {'ignored': 14331, 'no_valid': 8748, 'nonwater': 55470, 'water': 6999}}
0_0 {'C': 0.1, 'class_weight': 'none'} 0.5 eligible: True val IoU: 0.24232042699736447 fit: {'class_counts': {'0': 158771, '1': 16660}, 'converged': True, 'convergence_rule': 'gradient_tolerance_or_lbfgs_stop_before_max_iter', 'gradient_inf_norm': 9.748348929861095e-06, 'iterations': 129, 'objective': 0.035801202058792114, 'samples': 175431}
0_1 {'C': 0.1, 'class_weight': 'none'} 0.7 eligible: True val IoU: 0.31684167884819114 fit: {'class_counts': {'0': 158771, '1': 16660}, 'converged': True, 'convergence_rule': 'gradient_tolerance_or_lbfgs_stop_before_max_iter', 'gradient_inf_norm': 9.748348929861095e-06, 'iterations': 129, 'objective': 0.035801202058792114, 'samples': 175431}
1_0 {'C': 0.1, 'cl

## 4. Recuperación offline en otro runtime

Guarda el notebook **con salidas**; desconecta y elimina el runtime de Colab, crea otro con GPU. Ejecuta **solo** la celda de entorno de sección 1 y `recover` debajo. Conserva paquete/SHA/raíz idénticos. No vuelvas a ejecutar prepare/resume/inspección en esta segunda sesión; conserva sus salidas anteriores.

Se exige boot ID distinto, recarga offline de pesos, hashes de todos los artefactos, sin reentrenar ni recalibrar. Se repite extracción de features y segmentación por puntos del primer val por régimen con tolerancias prefijadas. La malla se reconstruye desde candidatos persistidos, sin repetir AMG/NMS. Esta prueba acredita recuperación del paquete en otro runtime; no una nueva campaña.


In [3]:
command([str(PYTHON), str(REPO / 'scripts/run_stage5.py'), 'recover', '--root', PERSIST_ROOT], 'recover.log')


fatal: not a git repository (or any of the parent directories): .git
No broken requirements found.
 
 test_canonical_hash (test_stage1.PersistenceTests.test_canonical_hash) ... ok
test_colab_temporary_root_blocked (test_stage1.PersistenceTests.test_colab_temporary_root_blocked) ... ok
test_each_identity_change_is_rejected (test_stage1.PersistenceTests.test_each_identity_change_is_rejected) ... ok
test_export_reports_absent_access_instead_of_claiming_authorization (test_stage1.PersistenceTests.test_export_reports_absent_access_instead_of_claiming_authorization) ... ok
test_export_uses_saved_identity_and_only_verified_json (test_stage1.PersistenceTests.test_export_uses_saved_identity_and_only_verified_json) ... ok
test_head_never_logs_token_or_follows_redirect (test_stage1.PersistenceTests.test_head_never_logs_token_or_follows_redirect) ... ok
test_incomplete_and_corrupt_generation_preserve_previous (test_stage1.PersistenceTests.test_incomplete_and_corrupt_generation_preserve_previous) .

## 5. Entrega para revisión y parada

Comparte este notebook descargado con salidas de ambas sesiones; `stage5/evidence/<run_id>.zip`, validation.json, recovery_validation.json, config.json y environment.json. Adjunta logs/failure.json si falló algo. No compartas tokens, pesos base ni datos crudos. Los shards voluminosos permanecen en Drive; el revisor pedirá unidades concretas si fueran necesarias.

El asistente verificará fuentes, linaje y aislamiento, hashes disponibles, selección por IoU de segmentación val, confusiones de todas las predicciones, geometría/puntos, semántica de malla y reconstrucción desde disco. Las features omitidas del ZIP no se presentarán como recalculadas por el revisor. Interpretará las salidas y actualizará decisiones/estado antes del cierre.

**Detener aquí.** No iniciar etapas 6–9 ni demo. La ejecución local sintética no sustituye esta evidencia GPU del usuario. Asistencia IA: Codex preparó implementación, pruebas y documentos; no inventó medidas ni resultados.


In [4]:
RECOVERY_RUNS = sorted(p for p in (Path(PERSIST_ROOT) / 'stage5/evidence').iterdir()
                       if p.is_dir() and (p / 'recovery_validation.json').is_file())
assert RECOVERY_RUNS, 'Compartir fallo y logs si recover no terminó.'
recovery = json.loads((RECOVERY_RUNS[-1] / 'recovery_validation.json').read_text())
print(json.dumps({'run_id': recovery['run_id'], 'status': recovery['status'],
    'different_runtime_boot': recovery['different_runtime_boot'], 'closure': recovery['closure'],
    'reconstruction': {r: v['disk_reconstruction'] for r, v in recovery['regimes'].items()},
    'evidence_zip': str(RECOVERY_RUNS[-1].with_suffix('.zip'))}, indent=2))


{
  "run_id": "20261029T194653Z_8073c068fa_ed6583db",
  "status": "computation_passed_pending_external_review",
  "different_runtime_boot": true,
  "closure": "pending_external_review_of_user_notebook_json_and_artifacts",
  "reconstruction": {
    "leave_paraguay_out": {
      "grid_from_persisted_candidates": true,
      "id": "Ghana_1033830",
      "label_reader_disabled_during_inference": true,
      "mask_disagreement_pixels": 0,
      "passed": true,
      "probability_tolerance": {
        "mask_disagreement_pixels": 0,
        "probability_atol": 1e-06,
        "probability_rtol": 1e-05
      },
      "scope": "first_val_chip"
    },
    "official": {
      "grid_from_persisted_candidates": true,
      "id": "Ghana_1033830",
      "label_reader_disabled_during_inference": true,
      "mask_disagreement_pixels": 0,
      "passed": true,
      "probability_tolerance": {
        "mask_disagreement_pixels": 0,
        "probability_atol": 1e-06,
        "probability_rtol": 1e-05
    